CICLO DI TRAINING IN PYTORCH

Dietro le quinte di pytorch.
In Pytorch dobbiamo dire alla rete ogni cosa

- La sequenza operativa: i 4 passi che permettono l'aggiornamento dei pesi
- Come cambia il modello quando smette di studiare ed inizia l'esame, inferenza e valutazione, gestire la modalità 'eval' e il contesto di disattivazione dei gradienti per i test
- Struttura dei Loop: come organizzare gerarchicamente l'addestramento attraverso epoche e mini-batch

La Coreografica del Training.
A differenza di altri algortimi che nascondono tutto sotto il tapeto (esempio Keras che nasconde il processo di addestramento dietro una singola funzione), PyTorch richiede una gestione esplicita di ogni fase.
Questo approccio offre un controllo totale su  ciò che accade ai dati ed ai gradienti, possiamo intervenire ogni istante nel processo di apprendimento.
Ogni iterazione di addestramento segue un ordine logico rigoroso che trasforma un input grezzo in un aggiornamento dei parametri del modello.

I 4 pilastri dell'addestramento
- Zero Grad: l'operazione optimizer.zero_grad() pulisce la memoria dei gradienti precedenti per evitare accumuli errati tra batch diversi.
- Forward Pass: il dato che attraversa la rete per produrre la risposta, passando i dati attraverso i layer definiti nella classe nn.Module
- Backwaord Pass: la rete riflette sul suo errore e calcola quanto ogni neurone ha colpa nella sbaglio
- Step: movimento finale, dove il modello aggiorna i pesi e fa un passo in più verso la verità.
Chi decide quanto è grande l'errore del modello? qui entra in gioco la Loss Function


Il ruolo della Loss Function
Immagina come un insegnante severo ma giusto, guarda la previsione della rete ed il risultato reale, restituiendo un numero che è la punizione per il modello, più è alto questo numero più la rete è confusa. Durante il trainign Pytorch costruisce un grafo computazionale, una sorta di ragnatela invisibile che collega ogni operazione, permettendo all'errore di rifluire indietro fino al primissimo layer.

Perchè azzerare i gradienti?
In PyTorch i gradienti vengono sommati per impostazione predefinita nel campo '.grad' di ogni tensore. Questo è utile per alcune architetture specifiche, ma disastroso nel training standard.
Se dimentichi Zero Grad PyTorch pensa che vuoi sommare il suggerimento di oggi, a quello di ieri, a quello dell'altro ieri, ecc. Il modello cercherebbe di correggersi basandosi non solo sull'errore attuale, ma sulla somma degli errori passati, portando a una divergenza immediata.

Training vs Evaluation
Cambiare il comportamento del modello
Una rete neurale non si comporta allo stesso modo quando sta imparando e quando viene testata. Alcuni layer, come il Dropout o la Batch Normalization, devono cambiare logica.
PyTorch gestisce queste transizioni attraverso flag di stato che informano ogni modulo interno sulla natura della operazione corrente.
PyTorch ha bisogno che noi gli diciamo esplicitamente in quale fase ci troviamo, alcuni componenti della rete devono spegnersi

Strumenti tecnici per fare questo switch
Strumenti per la fase di test
- model.train(): attiva i comportamenti specifici dell'addestramento (fase di studio), permettendo ad esempio al Dropout di spegnere neuroni casuali
- model.eval(): disabilita la regolarizzazione attiva e imposta i layer per fornire predizioni stabili e deterministiche (fase di test)
- torch.no_grad(): impedisce a PyTorch di costruire il grafo computazionale, risparmiando memoria durante il test. Immagina che la rete, per ogni pensiero, scriva un diario segreto, il grafo computazionale, per poter tornare indietro e correggersi, durante il testo questo diario non serve, la rete diventa più veloce e consuma molta meno energia.
- Inferenza: durante la fase di predizioe non serve calcolare i gradienti, quindi disattivarli accelera notevomente l'esecuzione

Layer sensibili allo stato.
Quando chiamiamo .eval, ed entriamo nella fase di test, il Dropout smette  di mascherare i neuroni e scala i pesi in modo che l'intera rete contribuisca alla predizione (il Dropout in addestramento spegne neuroni a caso per obbligare gli altri layer a lavorare di più).
In modalità test il batch normalization utilizza le medie e le varianze calcolate durante l'addestramento invece di quelle del batch corrente.
Dimenticare di impostare .eval() durante il test può portare a metriche di accuratezza instabili o peggiori del previsto.

Perchè disattivare il tracciamento fa risparmiare spazio
Risparmio risorte in test
Ogni operazione nel forward pass richiede memoria per salvare i valori intermedi necessari alla backpropagation. Se vuoi salvare solo il risultato finale, non serve il video dell'intera produzione
Se non intendiamo fare 'backward' questa memoria è sprecata.
Utilizzare torch.no_grad è fondamentale per poter validare modelli trandi su dataset estesi senza incorrere in errori di memoria esaurita

Loop di Addestramento
Organizzare il flusso nel tempo
L'addestramento di una rete è un processo iterativo che si ripete su due livelli: il passaggio su tutto il dataset e l'elaborazione di piccoli gruppi di dati (batch).
Dobbiamo strutturare il codice in modo da monitorare l'avanzamento e calcolare statistiche aggregate per ogni fase.
Per farlo dobbiamo gestire una serie di cicli for annidati

Gerarchia delle iterazioni
Epoche e Batch a confronto
- Epoch Loop: il ciclo esterno che stabilisce quante volte l'intero set di dati deve essere visto dal modello, giro completo su tutto il dataset.
- Batch Loop: il ciclo interno (al precedente) che itera sul DataLoader per processare piccoli frammenti del dataset alla volta. E' il singolo boccone di dati che diamo alla rete. 
- Accumulo Loss: solitamente si somma la perdita di ogni batch per calcolare la perdita media dell'epoca.
- Monitoraggio: la visualizzazione periodica dell'errore aiuta a capire se la convergenza sta avvenendo correttamente.

Alla fine di ogni batch calcoliamo i pesi, alla fine di ogni epoca calcoliamo la media dell'errore.
E' come se uno studente fa un mini testo alla fine di ogni pagina ed un esame alla fine di tutto il libro.

Il ruolo del DataLoader
Taglia il dataset in piccoli batch
Il DataLoader si occupa di mescolare i dati (shuffling) e dividerli in batch permettendo al loop di training di rimanere pulito.
A ogni passo del loop interno, riceviamo una coppia di tensori (input, target) pronti per essere passati al modello.
Il parametro Batch Size influenza la stabilità del gradiente: un batch piccolo è rumoroso ma veloce, un batch grande è stabile ma pesante. Scegliere la dimensione del batch è importante è la leva che usiamo per stabilizzare i gradienti
Ma dopo tanto lavoro come facciamo a capire se il modello sta imparando?

Convergenze e Statistiche
Osservare l'apprendimento
Al termine di ogni epoca, è buona pratica calcolare l'accuratezza e la perdita anche sul validation set, utilizzando la modalità eval vista in precedenza.
Se la curva scende allora stiamo andando bene, ma attenzione, bisogna confrontare le due curve (train ed eval)
Loss(eval)>Loss(train) -> Overfitting
Questo confronto permette di identificare immediatamente fenomeni di overfitting, dove la loss di training scende ma quella di validazione aumenta.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

#1.PREPARAZIONE DATI
#Generazione di 200 punti tra -5 e 5
X=torch.linspace(-5,5,200).view(-1,1)
#i layer lineari (nn.linear) si aspettano (n.righe,n.colonne), quindi porto a 200,1 (-1 indica di calcolarsi il n. di righe)

#target
y_target=torch.sin(X)+0.1*torch.randn(X.size())

#Creazione del DataLoader per gestire i batch durante l'addestramento
dataset=TensorDataset(X,y_target)
loader=DataLoader(dataset,batch_size=32,shuffle=True)
#stai dicendo dividi l'intero datasete in batch di 32 esempi, ogni esempio aggiorna i pesi
#esso dipende dalla dimensione del dataset

#2.MODELLO MULTI-STRATO
class DeepInspectorNet(nn.Module):  #creazione modello PyTorch
    def __init__(self):
        super().__init__() #inizializzo la classe padre
        #Definizione dei layer lineari
        self.fc1=nn.Linear(1,64)  #input 1 feature, output 64 neuroni (entrano 62 righe, date dal batch, per 1 feature, escono 32 righe per 64 feature)
        self.fc2=nn.Linear(64,32)
        self.out=nn.Linear(32,1) #il layer finale produce 1 valore (per 62 righe), quindi probabilmente regressione oppure output singolo contnuo, no classificazione multiclasse (avremmo nn.linear(32,n_classi))
        self.relu=nn.ReLU() #funzione di attivazione per introdurre non-linearità
        #lineare+lineare+lineare rimarrebbe lineare senza ReLu (relu introduce la non linearità), comportamento valori negativi=0 valori positivi=invariati
        #diventa: input-linear(1,64)-ReLu-linear(64,32)-ReLu-Linear(32,1)-Output

    def forward(self, x):
        #Passaggio dei dati attraverso la rete
        x=self.relu(self.fc1(x))
        x=self.relu(self.fc2(x))
        return self.out(x)        

model=DeepInspectorNet()
optimizer = optim.Adam(model.parameters(), lr=0.01) #Adam è il migliori in termini di velocità di risultato
criterion=nn.MSELoss() #errore quadratico medio (MSE), criterio per calcolo dell'errore

#3. CICLO DI TRAINING
epochs=100
print(f"{'Epoca':<10} | {'Loss Media':<10}")
print("-"*25)

for epoch in range(epochs):
    model.train()
    total_loss=0
    for batch_X, batch_y in loader:
        optimizer.zero_grad()          # Azzera i gradienti accumulati
        preds = model(batch_X)         # Calcola le predizioni
        loss = criterion(preds, batch_y) # Calcola l'errore
        loss.backward()                # Calcola i gradienti (Backpropagation)
        optimizer.step()               # Aggiorna i pesi del modello
        total_loss += loss.item()

    #Stampa l'andamento ogni 10 epoche
    if (epoch+1)%10==0:
        avg_loss=total_loss/len(loader)
        print(f"Epoca: {epoch} | media loss: {avg_loss}")

#4. GRAFICO FINALE
plt.figure(figsize=(10,6))

#Modalità valutazione per generare le predizioni finali
model.eval()
with torch.no_grad():  #per migliore gestione delle risorse
    final_pred=model(X)

#Visualizzazione dei dati reali rispetto alla curva appresa
plt.scatter(X.numpy(),y_target.numpy(),color="gray",alpha=0.5,label="Dati con Rumore")
plt.plot(X.numpy(),final_pred.numpy(),color="green",linewidth=3,label="Predizioni Modello")
plt.title("Risultato della Regressione (Approsimazione del Seno)")
plt.xlabel("Input(X)")
plt.ylabel("Output(y)")
plt.legend()
plt.grid(True,alpha=0.3)
plt.show()
print("\nAddestramento terminato - grafico visualizzato")


Epoca      | Loss Media
-------------------------
Epoca: 9 | media loss: 0.026052876641707762
Epoca: 19 | media loss: 0.025771783398730413
Epoca: 29 | media loss: 0.016141927002796104
Epoca: 39 | media loss: 0.015237042813428811
Epoca: 49 | media loss: 0.013441481760569982
Epoca: 59 | media loss: 0.011058877554855176
Epoca: 69 | media loss: 0.013483318899359022
Epoca: 79 | media loss: 0.013804862275719643
Epoca: 89 | media loss: 0.013609841199857848
Epoca: 99 | media loss: 0.010492976821426834


: 